# Tableau détaillé par point de données — `blob_nn_4x10-1` (gain vs baseline)

Ce notebook fusionne tous les `results.csv` (un par combo de bound strategy) et produit
**une ligne par point de données** (`data_index`, `target`) pour chaque combo — plus
d'agrégation/moyenne, chaque run individuel est conservé.

Colonnes produites :
`combo, strategy, l, u_idx, k, j_idx, data_index, target, certified, optimal_value, gain, LB_neuron1, UB_neuron1, LB_neuron2, UB_neuron2`

**Règles retenues** :
- `certified` = `optimal_value > 0` (par ligne).
- `results.csv` et `stable_actives_study.csv` sont tronqués à **78 lignes max** ; un
  combo avec **moins de 78 lignes** est **écarté**.
- **`gain`** = `optimal_value_combo − optimal_value_baseline` pour **ce point de
  données précis**, en appariant chaque ligne du combo à la ligne du **baseline
  `combo_0000`** ayant le **même `data_index` et le même `target`**.
  → **Hypothèse à confirmer** : `(data_index, target)` identifie une ligne de façon
  unique dans `results.csv`. Si ce n'est pas le cas, dites-le-moi et j'ajoute
  `epsilon` à la clé de jointure.
- **`LB_neuron1`/`UB_neuron1`/`LB_neuron2`/`UB_neuron2`** proviennent de
  `stable_actives_study.csv`, appariées par **`data_index`** uniquement (les bornes
  ne dépendent que de l'entrée, pas du `target`).
- Le baseline `combo_0000` apparaît lui aussi ligne par ligne, avec `gain = 0`.


In [7]:
import re
from pathlib import Path

import pandas as pd

# === CONFIGURATION ===

# Racine contenant tous les dossiers de runs (un par combo).
# Adaptez ce chemin à votre arborescence WSL, ex:
# BENCHMARK_ROOT = Path("/mnt/c/Users/vous/.../results/benchmark/blob_nn_4x10-1")
BENCHMARK_ROOT = Path("../results/benchmark/blob_nn_4x10-1")

# Nom du dossier de run, ex:
#   2026_07_08_15h28_19s_test__combo_0000__baseline__all_one_variable
#   2026_07_09_01h54_43s_test__combo_0001__product_0__l0_u0_k1_j0__composed
#   2026_07_14_14h56_49s_your_run_title__combo_0232__product_231__l2_u9_k3_j0__composed
RUN_DIR_RE = re.compile(r"combo_(\d+)__(.+)$")
PRODUCT_IDX_RE = re.compile(r"l(\d+)_u(\d+)_k(\d+)_j(\d+)")

# Un combo est gardé seulement s'il a au moins ce nombre de lignes utilisables ;
# au-delà, on tronque aux MAX_RUNS premières lignes ; en dessous, le combo est écarté.
MIN_RUNS = 0
MAX_RUNS = float("inf")

# Clé utilisée pour apparier chaque ligne d'un combo à la ligne correspondante du
# baseline (combo_0000), afin de calculer le gain.
RESULTS_MERGE_KEYS = ["data_index", "target"]

# Clé utilisée pour apparier les bornes LB/UB de stable_actives_study.csv
# (les bornes ne dépendent que de l'entrée, pas du target).
STABLE_MERGE_KEY = "data_index"


In [8]:
def parse_run_dir(dirname: str) -> dict:
    """Extrait combo_id, stratégie, et les indices (l,u,k,j) si présents."""
    m = RUN_DIR_RE.search(dirname)
    if not m:
        return None

    combo_id = int(m.group(1))
    rest = m.group(2)  # ex: "product_0__l0_u0_k1_j0__composed" ou "baseline__all_one_variable"

    idx_match = PRODUCT_IDX_RE.search(rest)
    if idx_match:
        l, u, k, j = map(int, idx_match.groups())
        strategy = rest.rsplit("__", 1)[-1]  # dernier segment = nom de la stratégie
    else:
        l = u = k = j = None
        strategy = "none" if combo_id == 0 else rest

    return {
        "combo": f"combo_{combo_id:04d}",
        "combo_id": combo_id,
        "strategy": strategy,
        "l": l, "u_idx": u, "k": k, "j_idx": j,
    }


def find_run_dirs(root: Path) -> list[dict]:
    """Retourne la liste des runs trouvés avec leurs métadonnées et chemins de fichiers."""
    if not root.exists():
        raise FileNotFoundError(f"Dossier introuvable: {root.resolve()}")

    runs = []
    for d in sorted(root.glob("*__combo_*")):
        if not d.is_dir():
            continue
        meta = parse_run_dir(d.name)
        if meta is None:
            print(f"[!] Nom de dossier non reconnu, ignoré: {d.name}")
            continue

        results_csv = d / "results.csv"
        stable_csv = d / "stable_actives_study.csv"

        if not results_csv.exists():
            print(f"[!] Pas de results.csv dans {d.name}")
            continue
        if not stable_csv.exists():
            print(f"[!] Pas de stable_actives_study.csv dans {d.name}")

        meta["results_csv"] = results_csv
        meta["stable_csv"] = stable_csv if stable_csv.exists() else None
        meta["dirname"] = d.name
        runs.append(meta)

    # Plusieurs dossiers peuvent partager le même combo_id (reruns à des timestamps
    # différents). On ne garde que le plus récent (le nom de dossier commence par un
    # timestamp AAAA_MM_JJ_HHhMMmSSs, donc trier les chaînes = trier chronologiquement).
    by_combo = {}
    for r in runs:
        cid = r["combo_id"]
        if cid not in by_combo or r["dirname"] > by_combo[cid]["dirname"]:
            by_combo[cid] = r

    n_dupes = len(runs) - len(by_combo)
    if n_dupes:
        counts = {}
        for r in runs:
            counts[r["combo_id"]] = counts.get(r["combo_id"], 0) + 1
        dupe_ids = sorted(cid for cid, n in counts.items() if n > 1)
        print(f"[!] {n_dupes} dossier(s) en double détecté(s) pour {len(dupe_ids)} combo(s), on garde le plus récent : {dupe_ids}")

    return sorted(by_combo.values(), key=lambda r: r["combo_id"])


runs = find_run_dirs(BENCHMARK_ROOT)
print(f"{len(runs)} run(s) trouvé(s):")
for r in runs:
    print(f"  {r['combo']} ({r['strategy']}) l={r['l']} u={r['u_idx']} k={r['k']} j={r['j_idx']}")


[!] Pas de results.csv dans 2026_07_10_13h31_53s_test__combo_0000__baseline__all_one_variable
[!] Pas de results.csv dans your_run_title__combo_0163__product_162__l2_u1_k4_j0__composed
[!] Pas de results.csv dans your_run_title__combo_0247__product_246__l3_u1_k4_j0__composed
[!] 19 dossier(s) en double détecté(s) pour 17 combo(s), on garde le plus récent : [0, 1, 2, 3, 4, 6, 8, 11, 14, 17, 18, 19, 20, 21, 22, 23, 172]
158 run(s) trouvé(s):
  combo_0000 (none) l=None u=None k=None j=None
  combo_0001 (composed) l=0 u=0 k=1 j=0
  combo_0002 (composed) l=0 u=0 k=1 j=1
  combo_0003 (composed) l=0 u=0 k=1 j=2
  combo_0004 (composed) l=0 u=0 k=2 j=0
  combo_0005 (composed) l=0 u=0 k=2 j=1
  combo_0006 (composed) l=0 u=0 k=2 j=2
  combo_0007 (composed) l=0 u=0 k=3 j=0
  combo_0008 (composed) l=0 u=0 k=3 j=1
  combo_0009 (composed) l=0 u=0 k=3 j=2
  combo_0010 (composed) l=0 u=0 k=4 j=0
  combo_0011 (composed) l=0 u=0 k=4 j=1
  combo_0012 (composed) l=0 u=0 k=4 j=2
  combo_0013 (composed) l=0 

In [9]:
def load_limited_csv(csv_path: Path, min_rows: int = MIN_RUNS, max_rows: int = MAX_RUNS) -> pd.DataFrame | None:
    """Charge un CSV, le tronque à max_rows lignes, ou renvoie None s'il en a moins que min_rows."""
    df = pd.read_csv(csv_path)
    n_original = len(df)
    if n_original < min_rows:
        return None
    if n_original > max_rows:
        df = df.iloc[:max_rows].reset_index(drop=True)
    assert len(df) <= max_rows, (
        f"BUG: {csv_path} a {len(df)} lignes après troncature (attendu <= {max_rows})"
    )
    return df


In [10]:
# === Baseline (combo_0000) ===
baseline_run = next((r for r in runs if r["combo_id"] == 0), None)
if baseline_run is None:
    raise RuntimeError("Aucun combo_0000 (baseline) trouvé — impossible de calculer le gain.")

baseline_results_df = load_limited_csv(baseline_run["results_csv"])
if baseline_results_df is None:
    raise RuntimeError(
        f"combo_0000 a moins de {MIN_RUNS} lignes utilisables — impossible de calculer un gain fiable."
    )

missing_keys = [k for k in RESULTS_MERGE_KEYS if k not in baseline_results_df.columns]
if missing_keys:
    raise RuntimeError(f"Colonnes de jointure manquantes dans results.csv du baseline: {missing_keys}")

print(f"Baseline: {baseline_run['combo']} ({baseline_run['strategy']}) — {len(baseline_results_df)} lignes")


Baseline: combo_0000 (none) — 4 lignes


In [11]:
def per_point_table(r: dict, baseline_results_df: pd.DataFrame) -> pd.DataFrame | None:
    """Construit une ligne par point de données pour un combo donné, avec gain et bornes LB/UB."""
    results_df = load_limited_csv(r["results_csv"])
    if results_df is None:
        return None

    # IMPORTANT: `data_index` est un compteur global qui continue de s'incrémenter
    # d'un combo à l'autre (ex: baseline = 0..76, combo_0001 = 921..999...), donc sa
    # valeur brute ne peut PAS servir à identifier "la même image" entre deux combos.
    # On apparie donc par POSITION relative : on trie chaque fichier par data_index
    # croissant (ordre de test), puis on compare le 1er point du combo au 1er point
    # du baseline, le 2e au 2e, etc.
    results_sorted = results_df.sort_values("data_index").reset_index(drop=True)
    results_sorted["point_rank"] = results_sorted.index

    baseline_sorted = baseline_results_df.sort_values("data_index").reset_index(drop=True)
    baseline_sorted["point_rank"] = baseline_sorted.index

    df = results_sorted[["data_index", "target", "optimal_value", "point_rank"]].copy()
    df["certified"] = results_sorted["optimal_value"] > 0

    # Gain point par point vs baseline (jointure sur la position, pas sur data_index)
    merged = df.merge(
        baseline_sorted[["point_rank", "optimal_value"]],
        on="point_rank",
        suffixes=("", "_baseline"),
        how="left",
    )
    if merged["optimal_value_baseline"].isna().any():
        n_missing = merged["optimal_value_baseline"].isna().sum()
        print(f"[!] {r['combo']}: {n_missing} ligne(s) sans correspondance dans le baseline (longueurs différentes ?)")
    merged["gain"] = merged["optimal_value"] - merged["optimal_value_baseline"]
    merged = merged.drop(columns=["optimal_value_baseline", "point_rank"])

    # Bornes LB/UB par data_index, depuis stable_actives_study.csv (même dossier,
    # donc même numérotation data_index — pas besoin de matching par position ici)
    stable_df = load_limited_csv(r["stable_csv"]) if r["stable_csv"] is not None else None
    lb1_col = f"LB_Layer_{r['l']}_Neuron_{r['u_idx']}" if r["l"] is not None else None
    ub1_col = f"UB_Layer_{r['l']}_Neuron_{r['u_idx']}" if r["l"] is not None else None
    lb2_col = f"LB_Layer_{r['k']}_Neuron_{r['j_idx']}" if r["k"] is not None else None
    ub2_col = f"UB_Layer_{r['k']}_Neuron_{r['j_idx']}" if r["k"] is not None else None

    for col_name, source_col in [
        ("LB_neuron1", lb1_col), ("UB_neuron1", ub1_col),
        ("LB_neuron2", lb2_col), ("UB_neuron2", ub2_col),
    ]:
        if stable_df is None or source_col is None or source_col not in stable_df.columns:
            merged[col_name] = float("nan")
        else:
            bounds = stable_df[[STABLE_MERGE_KEY, source_col]].rename(columns={source_col: col_name})
            merged = merged.merge(bounds, on=STABLE_MERGE_KEY, how="left")

    merged.insert(0, "combo", r["combo"])
    merged.insert(1, "strategy", r["strategy"])
    merged.insert(2, "l", r["l"])
    merged.insert(3, "u_idx", r["u_idx"])
    merged.insert(4, "k", r["k"])
    merged.insert(5, "j_idx", r["j_idx"])

    return merged


In [12]:
tables = []
skipped = []

for r in runs:
    t = per_point_table(r, baseline_results_df)
    if t is None:
        skipped.append(r["combo"])
        continue
    tables.append(t)

if skipped:
    print(f"{len(skipped)} combo(s) écarté(s) (moins de {MIN_RUNS} runs): {skipped}")

detail_df = pd.concat(tables, ignore_index=True)

# Vérification explicite : aucun combo ne doit dépasser MAX_RUNS lignes.
counts = detail_df.groupby("combo").size()
bad = counts[counts > MAX_RUNS]
if not bad.empty:
    raise AssertionError(f"Combos dépassant MAX_RUNS lignes après troncature: {bad.to_dict()}")

detail_df = detail_df[
    [
        "combo", "strategy", "l", "u_idx", "k", "j_idx",
        "data_index", "target", "certified", "optimal_value", "gain",
        "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2",
    ]
]
detail_df


[!] combo_0001: 74 ligne(s) sans correspondance dans le baseline (longueurs différentes ?)
[!] combo_0002: 74 ligne(s) sans correspondance dans le baseline (longueurs différentes ?)
[!] combo_0003: 74 ligne(s) sans correspondance dans le baseline (longueurs différentes ?)
[!] combo_0004: 74 ligne(s) sans correspondance dans le baseline (longueurs différentes ?)
[!] combo_0005: 74 ligne(s) sans correspondance dans le baseline (longueurs différentes ?)
[!] combo_0006: 74 ligne(s) sans correspondance dans le baseline (longueurs différentes ?)
[!] combo_0007: 74 ligne(s) sans correspondance dans le baseline (longueurs différentes ?)
[!] combo_0008: 74 ligne(s) sans correspondance dans le baseline (longueurs différentes ?)
[!] combo_0009: 74 ligne(s) sans correspondance dans le baseline (longueurs différentes ?)
[!] combo_0010: 74 ligne(s) sans correspondance dans le baseline (longueurs différentes ?)
[!] combo_0011: 74 ligne(s) sans correspondance dans le baseline (longueurs différentes ?)

,combo,strategy,l,u_idx,k,j_idx,data_index,target,certified,optimal_value,gain,LB_neuron1,UB_neuron1,LB_neuron2,UB_neuron2
0,combo_0000,none,None,None,None,None,0,NaN,False,-9.339218,0.0,NaN,NaN,NaN,NaN
1,combo_0000,none,None,None,None,None,1,NaN,False,-10.558940,0.0,NaN,NaN,NaN,NaN
2,combo_0000,none,None,None,None,None,2,NaN,True,2.044386,0.0,NaN,NaN,NaN,NaN
3,combo_0000,none,None,None,None,None,3,NaN,False,-0.025818,0.0,NaN,NaN,NaN,NaN
4,combo_0001,composed,0,0,1,0,921,NaN,False,NaN,NaN,0.482191,1.482191,-0.652311,0.415303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12482,combo_0330,composed,4,9,5,2,995,NaN,False,-11.537115,NaN,-1.330692,0.199566,-7.090365,70.517860
12483,combo_0330,composed,4,9,5,2,996,NaN,True,1.105619,NaN,-2.688242,-1.124644,-7.004410,49.300579
12484,combo_0330,composed,4,9,5,2,997,NaN,True,3.335567,NaN,-3.066478,-0.740262,-12.573580,-8.179303
12485,combo_0330,composed,4,9,5,2,998,NaN,True,2.335883,NaN,-2.868703,-0.776741,-11.508393,-1.407166


## Export (CSV, une ligne par point de données)

In [13]:
OUTPUT_CSV = "combo_detail_by_datapoint.csv"
detail_df.to_csv(OUTPUT_CSV, index=False)
print(f"Tableau exporté vers {OUTPUT_CSV} ({len(detail_df)} lignes)")


Tableau exporté vers combo_detail_by_datapoint.csv (12487 lignes)
